# Recommendation Model — score the new 100,000 customers\n\nTrains a **multi-class classifier** on the seed 100,000 to predict which of the 8 real `DimProduct` entries fits each customer (rather than the fixed one-product-per-type mapping the original generator used), then applies it to the new 100,000 and writes into `FactCustomerRecommendation`. `Score` and `PriorityScore` are separate regressors. `RecommendationType` is a direct pass-through of `CustomerType` (Next_Best_Product for Retail, Cross_Sell otherwise, same as the original generator - not modeled). `Channel` is randomly assigned, same as it effectively is in the original data (there's no real signal there to model).\n\n**Important, honest limitation specific to this notebook:** in the seed data, `ProductId` is *itself* a fixed, deterministic function of `CustomerType` alone (Corporate→3, Retail→4, SME→5 — only 3 of the 8 products ever actually appear). So the \"multi-class\" classifier here mostly has one real signal available to it (customer type) and will likely just relearn that same 3-way mapping with high confidence, not discover nuanced personalization — there's no way around this without richer historical recommendation-acceptance data, which doesn't exist yet.

In [1]:
from datetime import date

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from db_utils import bulk_insert, get_connection

FEATURE_COLS = [
    "Age", "Gender", "Region", "CustomerType", "Segment", "CustomerStatus",
    "Balance", "AccountType",
]
TODAY = date.today()

conn = get_connection()
seed = pd.read_sql(
    """
    SELECT c.CustomerId, c.Age, c.Gender, c.Region, c.CustomerType, c.Segment, c.CustomerStatus,
           a.Balance, a.AccountType,
           r.ProductId, r.Score, r.PriorityScore
    FROM dbo.DimCustomer c
    JOIN dbo.FactCustomerAccount a ON a.CustomerId = c.CustomerId
    JOIN dbo.FactCustomerRecommendation r ON r.CustomerId = c.CustomerId
    WHERE c.CustomerId <= 100000;
    """,
    conn,
)
products = pd.read_sql("SELECT ProductId, ProductName FROM dbo.DimProduct;", conn)
conn.close()

new_customers = pd.read_csv("data/new_customers_features.csv")
print("Seed:", seed.shape, " New:", new_customers.shape)
print(products)

/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_40637/2085144067.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seed = pd.read_sql(


Seed: (100000, 12)  New: (100000, 15)
   ProductId               ProductName
0          1           Classic Savings
1          2    Salary Current Account
2          3      Platinum Credit Card
3          4  SME Working Capital Loan
4          5      Retail Personal Loan
5          6    Treasury Fixed Deposit
6          7      Life Insurance Cover
7          8    Mobile Banking Premium


/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_40637/2085144067.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT ProductId, ProductName FROM dbo.DimProduct;", conn)


## 1. Product classifier

In [2]:
combined = pd.concat([seed[FEATURE_COLS], new_customers[FEATURE_COLS]], keys=["seed", "new"])
combined_encoded = pd.get_dummies(combined, drop_first=True)

X_seed = combined_encoded.loc["seed"].reset_index(drop=True)
X_new = combined_encoded.loc["new"].reset_index(drop=True)
y_product = seed["ProductId"].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_seed, y_product, test_size=0.2, random_state=42, stratify=y_product
)
eval_clf = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)
eval_clf.fit(X_train, y_train)
y_pred = eval_clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           3       1.00      1.00      1.00      6667
           4       1.00      1.00      1.00      6667
           5       1.00      1.00      1.00      6666

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



In [3]:
final_clf = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)
final_clf.fit(X_seed, y_product)
predicted_product = final_clf.predict(X_new)

pd.Series(predicted_product).value_counts()

4    69969
5    19988
3    10043
Name: count, dtype: int64

## 2. Score and PriorityScore regressors

In [4]:
def train_and_predict_score(target_col):
    y = seed[target_col].reset_index(drop=True)
    X_train, X_test, y_train, y_test = train_test_split(X_seed, y, test_size=0.2, random_state=42)
    eval_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
    eval_model.fit(X_train, y_train)
    y_pred = eval_model.predict(X_test)
    print(f"{target_col} - MAE:", round(mean_absolute_error(y_test, y_pred), 4),
          " R2:", round(r2_score(y_test, y_pred), 4))

    final_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
    final_model.fit(X_seed, y)
    return np.clip(final_model.predict(X_new), 0, 1).round(4)

predicted_score = train_and_predict_score("Score")
predicted_priority = train_and_predict_score("PriorityScore")

Score - MAE: 0.0739  R2: 0.031


PriorityScore - MAE: 0.0406  R2: 0.231


## 3. Assemble and write into the live warehouse

In [5]:
rng = np.random.default_rng(11)

results = pd.DataFrame({
    "CustomerId": new_customers["CustomerId"],
    "RecommendationType": np.where(new_customers["CustomerType"] == "Retail", "Next_Best_Product", "Cross_Sell"),
    "ProductId": predicted_product,
    "Score": predicted_score,
    "RecommendationDate": TODAY,
    "Channel": rng.choice(["Email", "SMS", "Mobile App"], size=len(new_customers)),
    "PriorityScore": predicted_priority,
})

results = results.merge(products, on="ProductId", how="left")
print(results["ProductName"].value_counts())
results.head()

ProductName
SME Working Capital Loan    69969
Retail Personal Loan        19988
Platinum Credit Card        10043
Name: count, dtype: int64


,CustomerId,RecommendationType,ProductId,Score,RecommendationDate,Channel,PriorityScore,ProductName
0,100001,Cross_Sell,5,0.6000,2026-07-23,Email,0.6877,Retail Personal Loan
1,100002,Next_Best_Product,4,0.5999,2026-07-23,Email,0.6814,SME Working Capital Loan
2,100003,Cross_Sell,5,0.5869,2026-07-23,Mobile App,0.6745,Retail Personal Loan
3,100004,Next_Best_Product,4,0.6014,2026-07-23,SMS,0.6495,SME Working Capital Loan
4,100005,Next_Best_Product,4,0.5990,2026-07-23,SMS,0.6495,SME Working Capital Loan


In [6]:
cols = ["CustomerId", "RecommendationType", "ProductId", "Score", "RecommendationDate", "Channel", "PriorityScore"]

conn = get_connection()
n = bulk_insert(conn, "dbo.FactCustomerRecommendation", cols, list(results[cols].itertuples(index=False, name=None)))
check = pd.read_sql(
    "SELECT COUNT(*) AS NewRecommendationRows FROM dbo.FactCustomerRecommendation WHERE CustomerId > 100000;", conn
)
conn.close()
print(f"Inserted {n:,} rows into FactCustomerRecommendation")
check

Inserted 100,000 rows into FactCustomerRecommendation


/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_40637/2335988370.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql(


,NewRecommendationRows
0,100000
